# Equipo 19 · Avance 3: evaluación de la máscara de El Meslouhi (2011)

Este notebook evalúa **solo** el método de máscara original de Endo-STTN, El Meslouhi et al. (2011), sobre los videos
de C3VDv2. No entrena nada, no usa GPU y no compara contra otros métodos.

| Sección | Contenido |
|---|---|
| 1 | Cómo funciona el método |
| 2 | Preparar Colab |
| 3 | El método paso a paso sobre frames reales |
| 4 | Cuánto marca: estadísticas por video y en el tiempo |
| 5 | Revisión visual: 50 frames al azar y los casos extremos |
| 6 | Hoja de revisión para el equipo y resumen |

## 1. Cómo funciona el método

Los reflejos especulares tienen el color de la luz (casi blanco), mientras que el tejido tiene su propio color
(rojizo). El método aprovecha esa diferencia. Es la traducción a Python de
`Endo-STTN/dataset_prep/SpecularDetectionMeslouhi2011.m` y `createSpecularMasks288.m`, aplicada al frame de 288×288:

1. **Realce.** Cada canal se multiplica por la "blancura" del píxel:
   $$s = \frac{\min(R,G,B)}{\max(R,G,B)}, \qquad (R', G', B') = s\cdot(R, G, B)$$
   En un píxel blanco $s \approx 1$ y casi no cambia; en tejido rojo $s$ es pequeño y el píxel se oscurece.
2. **Paso a CIE-XYZ** (sRGB, iluminante D65, igual que `rgb2xyz` de MATLAB), con $Y$ la luminancia.
3. **Comparación:** con la luminancia cromática $y = Y/(X+Y+Z)$ (cercana a 1/3 en colores neutros), un píxel es reflejo si
   $$Y \ge y$$
   Después del realce, solo los píxeles **claros y poco saturados** cumplen esta condición.
4. **Dilatación** de 1 px con un rombo, para cubrir el borde del reflejo.

**Qué esperar.** Debe ignorar los residuos amarillos o marrones, que tienen color, aunque sean claros. Puede marcar
tejido muy pálido e intensamente iluminado (por ejemplo, la mucosa rosada del ciego cerca de la cámara), porque
también es claro y poco saturado. Este notebook sirve para ver cuánto pasa eso en C3VDv2.

**Referencia:** A. El Meslouhi, M. Kardouchi, H. Allali, T. Gadi y Y. A. Benkaddour, *Automatic detection and
inpainting of specular reflections for colposcopic images*, Central European Journal of Computer Science, 2011.

## 2. Preparar Colab

No necesita GPU. Lee los ZIPs de `MyDrive/Equipo19/raw/` directo de Drive (sirve cualquier video: solo se usa `rgb/`).
Los resultados se guardan en `MyDrive/Equipo19/re_entrenar_endo/resultados/mascara_meslouhi/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, subprocess

# ---------- Ajusta si hace falta ----------
FRAMES_POR_VIDEO = 40   # frames equiespaciados por video para las estadísticas
N_REVISION = 50         # frames al azar para la revisión visual
DRIVE = Path("/content/drive/MyDrive/Equipo19")
REPO_URL = "https://github.com/Jkalks2025/Eliminaci-n-de-artefactos-en-im-genes-de-endoscop-a.git"
# ------------------------------------------

assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")
REPO = Path("/content/repo")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
assert (DRIVE / "raw").exists(), f"No encuentro {DRIVE / 'raw'}"
for z in (DRIVE / "raw").glob("*.zip"):
    if not (REPO / "datos" / "raw" / z.name).exists():
        os.symlink(z, REPO / "datos" / "raw" / z.name)
SALIDA = DRIVE / "re_entrenar_endo" / "resultados" / "mascara_meslouhi"

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO / "Avance3_ReEntrenar_Endo"))
import zipfile
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import endosttn_c3vd as ec
from endosttn_c3vd import c3vd

SALIDA.mkdir(parents=True, exist_ok=True)
MAGENTA = np.array([255, 0, 255])
def con_mascara(img, m, alfa=0.7):
    out = img.copy(); out[m] = ((1 - alfa) * out[m] + alfa * MAGENTA).astype(np.uint8); return out

videos = {z.stem: z for z in sorted(c3vd.DIR_RAW.glob("*.zip")) if z.stem not in c3vd.EXCLUIR}
print(f"{len(videos)} videos:", list(videos))
assert videos, f"No hay ZIPs en {c3vd.DIR_RAW}"

def leer_frames(ruta, n):
    """n frames equiespaciados de un video, a 288x288: lista de (índice, rgb)."""
    with zipfile.ZipFile(ruta) as zf:
        rutas = ec.frames_rgb(zf)
        sel = np.unique(np.linspace(0, len(rutas) - 1, min(n, len(rutas))).astype(int))
        return [(int(i), ec.frame_288(c3vd.leer_rgb(zf, rutas[i]))) for i in sel], len(rutas)

datos, total_frames = {}, {}
for v, ruta in videos.items():
    datos[v], total_frames[v] = leer_frames(ruta, FRAMES_POR_VIDEO)
    print(f"  {v}: {len(datos[v])} de {total_frames[v]} frames", flush=True)

## 3. El método paso a paso

Para un frame de cada uno de tres videos: el frame, la "blancura" $s$, el frame realzado, la diferencia $Y - y$ (positiva =
reflejo) y la máscara final. Así se ve qué condición decide en cada zona.

In [ ]:
def pasos(rgb):
    img = rgb.astype(np.float64) / 255
    mn, mx = img.min(-1), img.max(-1)
    s = np.divide(mn, mx, out=np.zeros_like(mn), where=mx > 0)
    realce = s[..., None] * img
    xyz = ec._srgb_a_xyz(realce)
    suma = xyz.sum(-1)
    y = np.divide(xyz[..., 1], suma, out=np.ones_like(suma), where=suma > 0)
    return s, realce, xyz[..., 1] - y

ejemplos = [(v, *datos[v][len(datos[v]) // 2]) for v in list(datos)[:3]]
fig, ax = plt.subplots(len(ejemplos), 5, figsize=(19, 4 * len(ejemplos)), squeeze=False)
for fila, (v, i, rgb) in zip(ax, ejemplos):
    s, realce, dif = pasos(rgb)
    m = ec.mascara_meslouhi(rgb)
    lim = max(abs(dif).max(), 1e-3)
    paneles = [(rgb, f"{v} · {i}", None), (s, "1. blancura s = min/max", "gray"),
               (np.clip(realce, 0, 1), "1. realce s·RGB", None), (dif, "3. Y − y  (rojo = reflejo)", "RdBu_r"),
               (con_mascara(rgb, m), f"4. máscara ({100 * m.mean():.2f}%)", None)]
    for a, (img, t, cmap) in zip(fila, paneles):
        kw = {"vmin": -lim, "vmax": lim} if cmap == "RdBu_r" else {}
        im = a.imshow(img, cmap=cmap, **kw); a.set_title(t, fontsize=10); a.axis("off")
        if cmap: plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.savefig(SALIDA / "pasos_meslouhi.png", dpi=90, bbox_inches="tight"); plt.show()

## 4. Cuánto marca

Por frame: porcentaje de la imagen marcado, número de manchas (regiones conectadas) y tamaño de la mancha más grande.
Un reflejo típico es pequeño y numeroso; una mancha muy grande suele ser tejido pálido o una zona sobreexpuesta y vale la
pena revisarla.

In [ ]:
filas = []
for v, frames in datos.items():
    version = v.rsplit("_", 1)[-1] if v.rsplit("_", 1)[-1].startswith("v") else "?"
    for i, rgb in frames:
        m = ec.mascara_meslouhi(rgb)
        n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
        areas = stats[1:, cv2.CC_STAT_AREA] if n > 1 else np.array([0])
        filas.append({"video": v, "version": version, "segmento": ec.segmento(v), "frame": i,
                      "marcado_pct": 100 * m.mean(), "manchas": n - 1,
                      "mancha_mayor_pct": 100 * areas.max() / m.size})
df = pd.DataFrame(filas)
df.to_csv(SALIDA / "meslouhi_por_frame.csv", index=False)
resumen = df.groupby("video").agg(frames=("frame", "size"), marcado_medio_pct=("marcado_pct", "mean"),
                                  marcado_mediana_pct=("marcado_pct", "median"), marcado_max_pct=("marcado_pct", "max"),
                                  manchas_medias=("manchas", "mean"), mancha_mayor_max_pct=("mancha_mayor_pct", "max"),
                                  frames_sin_reflejo=("marcado_pct", lambda x: int((x == 0).sum())))
resumen.to_csv(SALIDA / "meslouhi_resumen.csv")
display(resumen.round(2))
print(f"En todo: {df.marcado_pct.mean():.2f}% de la imagen marcado en promedio (mediana {df.marcado_pct.median():.2f}%).")
print("Referencia del Avance 1 (criterio HSV, 5 videos): ~1.2% en v1 y 0.3–0.5% en v2/v3.")

In [ ]:
colores = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#7a5cd6", "#d64f8a", "#3aa6b9", "#8c8c8c"]
fig, ax = plt.subplots(1, 2, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1.4, 1]})
for k, (v, d) in enumerate(df.groupby("video")):
    c = colores[k % len(colores)]
    ax[0].plot(d.frame / max(total_frames[v] - 1, 1), d.marcado_pct, "-o", ms=3, lw=1.5, color=c, label=v)
ax[0].set_xlabel("posición en el video (0 = inicio, 1 = final)"); ax[0].set_ylabel("% de la imagen marcado")
ax[0].set_title("Máscara a lo largo de cada video"); ax[0].grid(alpha=0.3)
ax[0].legend(fontsize=8, ncol=2, frameon=False)
orden = resumen.sort_values("marcado_medio_pct").index
cajas = [df[df.video == v].marcado_pct for v in orden]
try:
    ax[1].boxplot(cajas, orientation="horizontal", widths=0.6)   # matplotlib >= 3.10
except TypeError:
    ax[1].boxplot(cajas, vert=False, widths=0.6)
ax[1].set_yticks(range(1, len(orden) + 1)); ax[1].set_yticklabels(orden, fontsize=8)
ax[1].set_xlabel("% de la imagen marcado por frame"); ax[1].set_title("Distribución por video"); ax[1].grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.savefig(SALIDA / "meslouhi_estadisticas.png", dpi=110, bbox_inches="tight"); plt.show()

## 5. Revisión visual

**50 frames al azar** de todos los videos, con la máscara en magenta. Al revisarlos conviene fijarse en:
- ¿Cubre los puntos y franjas blancas brillantes, incluido su borde?
- ¿Marca tejido sano pálido, residuos, espuma o el borde negro de la imagen?
- ¿Se le escapan reflejos tenues?

Después, los **casos extremos**: los 6 frames con más área marcada, que es donde más probablemente haya falsos positivos,
con el frame original al lado.

In [ ]:
rng = np.random.default_rng(0)
todos = [(v, i, rgb) for v, frames in datos.items() for i, rgb in frames]
revision = [todos[k] for k in sorted(rng.choice(len(todos), size=min(N_REVISION, len(todos)), replace=False))]
cols = 10
filas_fig = int(np.ceil(len(revision) / cols))
fig, ax = plt.subplots(filas_fig, cols, figsize=(2.2 * cols, 2.35 * filas_fig), squeeze=False)
for a, (n, (v, i, rgb)) in zip(ax.ravel(), enumerate(revision, 1)):
    m = ec.mascara_meslouhi(rgb)
    a.imshow(con_mascara(rgb, m)); a.set_title(f"#{n} {v} · {i}\n{100 * m.mean():.1f}%", fontsize=6.5)
for a in ax.ravel(): a.axis("off")
plt.suptitle(f"{len(revision)} frames al azar · El Meslouhi (magenta)", y=1.0); plt.tight_layout()
plt.savefig(SALIDA / "revision_meslouhi.png", dpi=80, bbox_inches="tight"); plt.show()

In [ ]:
extremos = df.sort_values("marcado_pct", ascending=False).head(6)
fig, ax = plt.subplots(2, len(extremos), figsize=(3.3 * len(extremos), 6.8), squeeze=False)
for j, (_, f) in enumerate(extremos.iterrows()):
    rgb = dict(datos[f.video])[f.frame]
    ax[0, j].imshow(rgb); ax[0, j].set_title(f"{f.video} · {f.frame}", fontsize=8)
    ax[1, j].imshow(con_mascara(rgb, ec.mascara_meslouhi(rgb))); ax[1, j].set_title(f"{f.marcado_pct:.1f}% marcado", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.suptitle("Frames con más área marcada (arriba original, abajo máscara)", y=1.0); plt.tight_layout()
plt.savefig(SALIDA / "meslouhi_extremos.png", dpi=90, bbox_inches="tight"); plt.show()

## 6. Hoja de revisión y resumen

Se guarda `hoja_revision_meslouhi.csv` con los frames de la revisión visual (mismo número `#` que en la figura) y columnas
vacías para que el equipo anote, frame por frame, si la máscara es correcta. Con eso se puede reportar al Dr. Ricardo
un porcentaje de frames aceptables, junto con las estadísticas de la sección 4.

In [ ]:
hoja = pd.DataFrame([{"#": n, "video": v, "frame": i, "marcado_pct": round(100 * ec.mascara_meslouhi(rgb).mean(), 2),
                      "cubre_reflejos (si/parcial/no)": "", "marca_otra_cosa (no/tejido/residuo/borde/otro)": "",
                      "comentario": ""} for n, (v, i, rgb) in enumerate(revision, 1)])
hoja.to_csv(SALIDA / "hoja_revision_meslouhi.csv", index=False)
with open(SALIDA / "resumen_meslouhi.md", "w") as f:
    f.write("# Evaluación de la máscara de El Meslouhi (2011) en C3VDv2\n\n")
    f.write(f"{len(videos)} videos, {FRAMES_POR_VIDEO} frames equiespaciados por video, a 288x288.\n\n")
    f.write(resumen.round(2).to_markdown() + "\n\n")
    f.write(f"Promedio general: {df.marcado_pct.mean():.2f}% de la imagen marcado (mediana {df.marcado_pct.median():.2f}%).\n\n")
    f.write("Figuras: pasos_meslouhi.png, meslouhi_estadisticas.png, revision_meslouhi.png, meslouhi_extremos.png. "
            "Revisión manual: hoja_revision_meslouhi.csv.\n")
print("Resultados ->", SALIDA)
print(sorted(p.name for p in SALIDA.iterdir()))